<a href="https://www.kaggle.com/code/alabiabiodun/sit-model-with-uncertainty-analysis?scriptVersionId=350294232" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# SIT Model with Uncertainty Analysis

## HIV transmission across linked population groups in Nigeria

**Original model:** 2013  
**Notebook edited:** 16 September 2026  

This notebook documents a mathematical model developed in the context of Nigeria's national HIV response. It follows interactions among female sex workers, their clients, general males, and general females. The model represents how transmission, testing, treatment, condom use, sexually transmitted infections, and treatment failure can shape HIV prevalence over time.

The original equations and parameter ranges are retained. The projection period has been extended from 2020 to 2030 to show the path produced when the same assumptions continue. The extension is a scenario projection, not a forecast of Nigeria's current epidemic.

## Summary

This notebook runs 100 uncertainty simulations from 2000 to 2030. Under the retained assumptions, average prevalence falls from 37.54 to 2.84 percent among female sex workers, from 10.74 to 0.63 percent among clients, from 5.69 to 0.30 percent among general males, and from 4.73 to 0.38 percent among general females.

All ten observed prevalence points are above the corresponding average model estimates. Mean absolute error is 9.86 percentage points for female sex workers and ranges from 1.99 to 2.38 percentage points for the other groups. The model is therefore more useful for demonstrating transmission pathways and intervention logic than for estimating current prevalence. Publicly available national indicators are included as context and are not used to calibrate the population-specific model.

## Historical context

The archival *Report on the Final Mathematical Modelling Training Course* (National Agency for the Control of AIDS, 2013) records that the National Agency for the Control of AIDS organized the training for staff and partners involved in Nigeria's HIV response. The University of New South Wales delivered the programme through an agreement with the World Bank. The final course took place in Lagos from 18 to 22 November 2013.

The objectives below are reproduced as written in the report:

> To train a core group that will support the modeling need in the National Response to HIV/AIDS.
>
> To developed and built the capacity of participants on mathematical modeling and its application in the management of HIV/AIDS routine data.
>
> To introduce participants to advanced techniques used in mathematical modeling of infectious disease
>
> Participants to complete and present findings from their group project.

The report also proposed a mathematical modelling group whose outputs would feed into national plans and provide continuous support to the National Technical Working Group. This notebook relates to the author's subsequent six-month engagement with the Technical Working Group on Mathematical Modelling of HIV in Nigeria. The work examined how interactions among population groups and intervention assumptions could affect HIV prevalence and programme planning.

**Archival source:** National Agency for the Control of AIDS, *Report on the Final Mathematical Modelling Training Course*, November 2013.

## What the SIT model represents

SIT refers to a compartmental structure centred on people who are **Susceptible**, **Infected**, or receiving **Treatment**. The implementation expands the infected and treatment stages so that programme pathways can be represented more clearly.

| Code | Model state | Meaning |
|---|---|---|
| `S` | Susceptible | Not living with HIV and at risk of infection |
| `U` | Undiagnosed | Living with HIV but not yet diagnosed |
| `D` | Diagnosed | Diagnosed with HIV but not currently receiving treatment |
| `T` | Treated | Receiving antiretroviral therapy |
| `F` | Failed treatment | Treatment is no longer effective in the model |

The states are applied to five linked groups: female sex workers without a sexually transmitted infection, female sex workers with a sexually transmitted infection, clients of female sex workers, general males, and general females.

Transmission moves people from `S` to `U`. Testing moves people from `U` to `D`. Treatment initiation moves people from `D` to `T`. Nonadherence can move people from `T` back to `D`, while treatment failure moves people from `T` to `F`. Births, natural mortality, HIV related mortality, condom protection, antiretroviral effectiveness, and sexually transmitted infection status also affect the yearly transitions.

## Modelling approach and assumptions

The model uses annual difference equations. Each uncertainty simulation draws one value from every parameter range using a uniform distribution, then projects the population states from 2000 through 2030.

Key assumptions retained from the original 2013 code are:

1. Parameters are sampled independently from uniform ranges.
2. Population mixing is represented through the specified links among female sex workers, clients, general males, and general females.
3. Condom use and treatment reduce transmission risk according to their sampled effectiveness.
4. Sexually transmitted infection increases susceptibility among affected female sex workers.
5. The displayed simulation band is the minimum to maximum across 100 runs. It is not a statistical confidence interval.
6. Observed prevalence points are sparse and are used only for a descriptive calibration comparison.

The original parameter values and equations are preserved. Code organization, output handling, and visualization have been modernized for Python 3 and notebook use.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from IPython.display import display

RANDOM_SEED = 0
UNCERTAINTY_SIMULATIONS = 100
SCENARIOS = [0]
YEARS = np.arange(2000, 2031)

np.random.seed(RANDOM_SEED)
pd.options.display.float_format = "{:,.2f}".format


plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#333333",
    "axes.labelcolor": "#222222",
    "axes.titleweight": "semibold",
    "font.size": 11,
    "grid.color": "#D9D9D9",
    "grid.linestyle": "--",
    "grid.linewidth": 0.7,
})

print(f"Simulation period: {YEARS[0]} to {YEARS[-1]}")
print(f"Uncertainty simulations: {UNCERTAINTY_SIMULATIONS}")

## Public HIV indicators for Nigeria

National indicators provide context for the historical model. They are shown separately because their population definitions, methods, and reporting years differ from the population-specific observations used for calibration.

The Nigeria HIV/AIDS Indicator and Impact Survey reported adult prevalence of 1.4 percent for ages 15 to 49 in 2018 (UNAIDS, 2019). More recent public estimates place adult prevalence at 1.3 percent in 2023 (United States Centers for Disease Control and Prevention, 2025) and 1.2 percent in 2024 (World Bank, 2025). Public reporting also provides treatment, mortality, and incidence measures that can support a future recalibration of this model.

Sources were accessed on 16 September 2026: [UNAIDS release on the 2018 survey](https://www.unaids.org/en/resources/presscentre/pressreleaseandstatementarchive/2019/march/20190314_nigeria), [CDC Nigeria HIV and tuberculosis overview](https://www.cdc.gov/global-hiv-tb/php/where-we-work/nigeria.html), [World Bank adult HIV prevalence indicator](https://data.worldbank.org/indicator/SH.DYN.AIDS.ZS?locations=NG), and [UNAIDS Nigeria country profile](https://www.unaids.org/en/regionscountries/countries/nigeria).

In [ ]:
public_indicators = pd.DataFrame([
    {
        "Indicator": "Adult HIV prevalence, ages 15–49",
        "Value": "1.4%",
        "Reference year": 2018,
        "Source": "Nigeria HIV/AIDS Indicator and Impact Survey, released by UNAIDS",
    },
    {
        "Indicator": "Estimated adult HIV prevalence, ages 15–49",
        "Value": "1.3%",
        "Reference year": 2023,
        "Source": "United States Centers for Disease Control and Prevention",
    },
    {
        "Indicator": "Adults receiving antiretroviral therapy",
        "Value": "1,690,291",
        "Reference year": 2023,
        "Source": "United States Centers for Disease Control and Prevention",
    },
    {
        "Indicator": "Estimated HIV deaths, age 15 and older",
        "Value": "30,000",
        "Reference year": 2023,
        "Source": "United States Centers for Disease Control and Prevention",
    },
    {
        "Indicator": "Estimated adult HIV prevalence, ages 15–49",
        "Value": "1.2%",
        "Reference year": 2024,
        "Source": "World Bank World Development Indicators, based on UNAIDS estimates",
    },
    {
        "Indicator": "HIV incidence, adults ages 15–49",
        "Value": "0.44 per 1,000",
        "Reference year": 2024,
        "Source": "UNAIDS Nigeria country profile",
    },
])

display(public_indicators.style.hide(axis="index"))

## Parameter ranges

In [ ]:
uncertainty_simulations = UNCERTAINTY_SIMULATIONS
scenarios = SCENARIOS
n_scenarios = len(scenarios)
years = YEARS
run_time = len(years)

# Demography and mortality
b_range = np.array([0.039, 0.040])
d_range = np.array([0.0135, 0.0140])
d_hiv_range = np.array([0.36, 0.38])
d_art_range = np.array([0.032, 0.036])
clientmixing = 0

# Transmission and population mixing
calibration_beta = 9
calibration_sw = 1
beta_mf_range = calibration_beta * np.array([0.0010, 0.0012])
beta_fm_range = 1.2 * calibration_beta * np.array([0.0005, 0.0006])
delta_range = np.array([2.0, 4.0])
n_cl_sw_range = calibration_sw * np.array([700.0, 800.0])
n_cl_gf_range = np.array([0.8, 1.0])
n_gm_gf_range = np.array([50.0, 60.0])
p_cl_sw_range = np.array([0.85, 0.96])
p_cl_gf_range = np.array([0.60, 0.65])
p_gm_gf_range = np.array([0.60, 0.65])

# Testing, treatment, adherence, and prevention
phi_sw_range = np.array([0.45, 0.55])
phi_cl_range = np.array([0.45, 0.55])
phi_gm_range = np.array([0.45, 0.55])
phi_gf_range = np.array([0.45, 0.55])
pi_sw_range = np.array([0.375, 0.45])
pi_cl_range = np.array([0.05, 0.15])
pi_gm_range = np.array([0.05, 0.15])
pi_gf_range = np.array([0.05, 0.15])
test_sw_range = np.array([0.562, 0.719])
test_op_range = np.array([0.200, 0.350])
zeta_range = np.array([0.03, 0.05])
alpha_range = np.array([0.0010, 0.0012])
treat_sti_range = np.array([0.60, 0.67])
e_art_range = np.array([0.62, 0.73])
e_condom_range = np.array([0.69, 0.87])

## Initial conditions and observed prevalence

In [ ]:
# Susceptible populations
Ssw_0 = np.array([33465, 35138])
Ssw_i_0 = np.array([6222, 6533])
Scl_0 = 15 * np.array([148633, 156064])
Sgm_0 = np.array([20855625, 21898406])
Sgf_0 = np.array([20917911, 21963806])

# Undiagnosed HIV
Usw_0 = 0.3 * 2 * Ssw_0
Usw_i_0 = 0.3 * 2 * Ssw_i_0
Ucl_0 = 0.12 * Scl_0
Ugf_0 = 0.05 * Sgm_0
Ugm_0 = 0.06 * Sgf_0

# Diagnosed, treated, and failed treatment states begin at zero
Dsw_0 = Dsw_i_0 = Dcl_0 = Dgf_0 = Dgm_0 = np.array([0.0, 0.0])
Tsw_0 = Tsw_i_0 = Tcl_0 = Tgf_0 = Tgm_0 = np.array([0.0, 0.0])
Fsw_0 = Fsw_i_0 = Fcl_0 = Fgf_0 = Fgm_0 = np.array([0.0, 0.0])

# Observed prevalence percentages retained from the legacy model
prev_data_sw = np.array([[2007, 33.6], [2010, 24.3]])
prev_data_cl = np.array([[2010, 5.3], [2012, 5.6]])
prev_data_gm = np.array([[2007, 3.5], [2010, 4.1], [2012, 3.3]])
prev_data_gf = np.array([[2007, 3.6], [2010, 4.2], [2012, 3.5]])

observed_prevalence = {
    "Female sex workers": prev_data_sw,
    "Clients": prev_data_cl,
    "General males": prev_data_gm,
    "General females": prev_data_gf,
}

display(pd.concat(
    [pd.DataFrame(values, columns=["Year", "Observed prevalence (%)"]).assign(Population=name)
     for name, values in observed_prevalence.items()],
    ignore_index=True,
)[["Population", "Year", "Observed prevalence (%)"]])

## Model state arrays

In [ ]:
shape = (uncertainty_simulations, run_time, n_scenarios)

Ssw, Usw, Dsw, Tsw, Fsw, Nsw = [np.zeros(shape) for _ in range(6)]
Ssw_i, Usw_i, Dsw_i, Tsw_i, Fsw_i, Nsw_i = [np.zeros(shape) for _ in range(6)]
Scl, Ucl, Dcl, Tcl, Fcl, Ncl = [np.zeros(shape) for _ in range(6)]
Sgm, Ugm, Dgm, Tgm, Fgm, Ngm = [np.zeros(shape) for _ in range(6)]
Sgf, Ugf, Dgf, Tgf, Fgf, Ngf = [np.zeros(shape) for _ in range(6)]

prevalences_SW_Combined = np.zeros(shape)
prevalences_SW = np.zeros(shape)
prevalences_SW_I = np.zeros(shape)
prevalences_CL = np.zeros(shape)
prevalences_GM = np.zeros(shape)
prevalences_GF = np.zeros(shape)

print(f"Allocated {shape[0]:,} simulations × {shape[1]} years × {shape[2]} scenario.")

## Uncertainty simulation

Each run samples the original parameter ranges, initializes the five population groups, calculates the force of infection across linked groups, and updates the five HIV programme states once per year.

In [ ]:
for k in range(uncertainty_simulations):

    b           = b_range[0]                + (b_range[1]               - b_range[0])               * np.random.rand()
    d           = d_range[0]                + (d_range[1]               - d_range[0])               * np.random.rand()
    d_hiv       = d_hiv_range[0]            + (d_hiv_range[1]           - d_hiv_range[0])           * np.random.rand()
    d_art       = d_art_range[0]            + (d_art_range[1]           - d_art_range[0])           * np.random.rand()
    beta_mf     = beta_mf_range[0]          + (beta_mf_range[1]         - beta_mf_range[0])         * np.random.rand()
    beta_fm     = beta_fm_range[0]          + (beta_fm_range[1]         - beta_fm_range[0])         * np.random.rand()
    delta       = delta_range[0]            + (delta_range[1]           - delta_range[0])           * np.random.rand()
    n_cl_sw     = n_cl_sw_range[0]          + (n_cl_sw_range[1]         - n_cl_sw_range[0])         * np.random.rand()
    n_cl_gf     = n_cl_gf_range[0]          + (n_cl_gf_range[1]         - n_cl_gf_range[0])         * np.random.rand()
    n_gm_gf     = n_gm_gf_range[0]          + (n_gm_gf_range[1]         - n_gm_gf_range[0])         * np.random.rand()
    p_cl_sw     = (p_cl_sw_range[0]         + (p_cl_sw_range[1]         - p_cl_sw_range[0])         * np.random.rand()) * np.ones(run_time)
    p_cl_gf     = (p_cl_gf_range[0]         + (p_cl_gf_range[1]         - p_cl_gf_range[0])         * np.random.rand()) * np.ones(run_time)
    p_gm_gf     = (p_gm_gf_range[0]         + (p_gm_gf_range[1]         - p_gm_gf_range[0])         * np.random.rand()) * np.ones(run_time)
    phi_sw      = phi_sw_range[0]           + (phi_sw_range[1]          - phi_sw_range[0])          * np.random.rand()
    phi_cl      = phi_cl_range[0]           + (phi_cl_range[1]          - phi_cl_range[0])          * np.random.rand()
    phi_gm      = phi_gm_range[0]           + (phi_gm_range[1]          - phi_gm_range[0])          * np.random.rand()
    phi_gf      = phi_gf_range[0]           + (phi_gf_range[1]          - phi_gf_range[0])          * np.random.rand()
    pi_sw       = pi_sw_range[0]            + (pi_sw_range[1]           - pi_sw_range[0])           * np.random.rand()
    pi_cl       = pi_cl_range[0]            + (pi_cl_range[1]           - pi_cl_range[0])           * np.random.rand()
    pi_gm       = pi_gm_range[0]            + (pi_gm_range[1]           - pi_gm_range[0])           * np.random.rand()
    pi_gf       = pi_gf_range[0]            + (pi_gf_range[1]           - pi_gf_range[0])           * np.random.rand()
    test_sw     = test_sw_range[0]          + (test_sw_range[1]         - test_sw_range[0])         * np.random.rand()
    test_op     = test_op_range[0]          + (test_op_range[1]         - test_op_range[0])         * np.random.rand()
    zeta        = zeta_range[0]             + (zeta_range[1]            - zeta_range[0])            * np.random.rand()
    alpha       = alpha_range[0]            + (alpha_range[1]           - alpha_range[0])           * np.random.rand()
    treat_sti   = treat_sti_range[0]        + (treat_sti_range[1]       - treat_sti_range[0])       * np.random.rand()
    e_art       = e_art_range[0]            + (e_art_range[1]           - e_art_range[0])           * np.random.rand()
    e_condom    = e_condom_range[0]         + (e_condom_range[1]        - e_condom_range[0])        * np.random.rand()

    for sc in range(n_scenarios):

        for j in range(run_time):

            if j == 0:

                Ssw[k,j,sc]        = Ssw_0[0]   + (Ssw_0[1]   - Ssw_0[0])   * np.random.rand()
                Ssw_i[k,j,sc]      = Ssw_i_0[0] + (Ssw_i_0[1] - Ssw_i_0[0]) * np.random.rand()
                Scl[k,j,sc] 	     = Scl_0[0]   + (Scl_0[1]   - Scl_0[0])   * np.random.rand()
                Sgm[k,j,sc]        = Sgm_0[0]   + (Sgm_0[1]   - Sgm_0[0])   * np.random.rand()
                Sgf[k,j,sc]        = Sgf_0[0]   + (Sgf_0[1]   - Sgf_0[0])   * np.random.rand()
                Usw[k,j,sc]        = Usw_0[0]   + (Usw_0[1]   - Usw_0[0])   * np.random.rand()
                Usw_i[k,j,sc]      = Usw_i_0[0] + (Usw_i_0[1] - Usw_i_0[0]) * np.random.rand()
                Ucl[k,j,sc]        = Ucl_0[0]   + (Ucl_0[1]   - Ucl_0[0])   * np.random.rand()
                Ugm[k,j,sc]        = Ugm_0[0]   + (Ugm_0[1]   - Ugm_0[0])   * np.random.rand()
                Ugf[k,j,sc]        = Ugf_0[0]   + (Ugf_0[1]   - Ugf_0[0])   * np.random.rand()
                Dsw[k,j,sc]        = Dsw_0[0]   + (Dsw_0[1]   - Dsw_0[0])   * np.random.rand()
                Dsw_i[k,j,sc]      = Dsw_i_0[0] + (Dsw_i_0[1] - Dsw_i_0[0]) * np.random.rand()
                Dcl[k,j,sc]        = Dcl_0[0]   + (Dcl_0[1]   - Dcl_0[0])   * np.random.rand()
                Dgm[k,j,sc]        = Dgm_0[0]   + (Dgm_0[1]   - Dgm_0[0])   * np.random.rand()
                Dgf[k,j,sc]        = Dgf_0[0]   + (Dgf_0[1]   - Dgf_0[0])   * np.random.rand()
                Tsw[k,j,sc]        = Tsw_0[0]   + (Tsw_0[1]   - Tsw_0[0])   * np.random.rand()
                Tsw_i[k,j,sc]      = Tsw_i_0[0] + (Tsw_i_0[1] - Tsw_i_0[0]) * np.random.rand()
                Tcl[k,j,sc]        = Tcl_0[0]   + (Tcl_0[1]   - Tcl_0[0])   * np.random.rand()
                Tgm[k,j,sc]        = Tgm_0[0]   + (Tgm_0[1]   - Tgm_0[0])   * np.random.rand()
                Tgf[k,j,sc]        = Tgf_0[0]   + (Tgf_0[1]   - Tgf_0[0])   * np.random.rand()
                Fsw[k,j,sc]        = Fsw_0[0]   + (Fsw_0[1]   - Fsw_0[0])   * np.random.rand()
                Fsw_i[k,j,sc]      = Fsw_i_0[0] + (Fsw_i_0[1] - Fsw_i_0[0]) * np.random.rand()
                Fcl[k,j,sc]        = Fcl_0[0]   + (Fcl_0[1]   - Fcl_0[0])   * np.random.rand()
                Fgm[k,j,sc]        = Fgm_0[0]   + (Fgm_0[1]   - Fgm_0[0])   * np.random.rand()
                Fgf[k,j,sc]        = Fgf_0[0]   + (Fgf_0[1]   - Fgf_0[0])   * np.random.rand()

                Nsw[k,j,sc]        = Ssw[k,j,sc]   + Usw[k,j,sc]   + Dsw[k,j,sc]   + Tsw[k,j,sc]   + Fsw[k,j,sc]
                Nsw_i[k,j,sc]      = Ssw_i[k,j,sc] + Usw_i[k,j,sc] + Dsw_i[k,j,sc] + Tsw_i[k,j,sc] + Fsw_i[k,j,sc]
                Ncl[k,j,sc]        = Scl[k,j,sc]   + Ucl[k,j,sc]   + Dcl[k,j,sc]   + Tcl[k,j,sc]   + Fcl[k,j,sc]
                Ngm[k,j,sc]        = Sgm[k,j,sc]   + Ugm[k,j,sc]   + Dgm[k,j,sc]   + Tgm[k,j,sc]   + Fgm[k,j,sc]
                Ngf[k,j,sc]        = Sgf[k,j,sc]   + Ugf[k,j,sc]   + Dgf[k,j,sc]   + Tgf[k,j,sc]   + Fgf[k,j,sc]

            else:

                i = j - 1

                clientsmove = np.array([Scl[k,i,sc],Ucl[k,i,sc],Dcl[k,i,sc],Tcl[k,i,sc],Fcl[k,i,sc]]) * clientmixing
                totalmalemove = np.sum(clientsmove)
                malesmove = np.array([Sgm[k,i,sc],Ugm[k,i,sc],Dgm[k,i,sc],Tgm[k,i,sc],Fgm[k,i,sc]]) * (totalmalemove/Ngm[k,i,sc])
                clientrecylce = malesmove - clientsmove

                assert abs(totalmalemove-np.sum(malesmove))<1, 'Clients not recyling properly'

                force_of_infection_gm = \
                ((Ugf[k,i,sc]+Dgf[k,i,sc]+Fgf[k,i,sc])/Ngf[k,i,sc]) * (1 - (1-(1-e_condom)*beta_fm)**(n_gm_gf*p_gm_gf[i])           * (1-beta_fm)**(n_gm_gf*(1-p_gm_gf[i]))) + \
                (Tgf[k,i,sc]                         /Ngf[k,i,sc]) * (1 - (1-(1-e_art)*(1-e_condom)*beta_fm)**(n_gm_gf*p_gm_gf[i]) * (1-(1-e_art)*beta_fm)**(n_gm_gf*(1-p_gm_gf[i])))

                force_of_infection_gf = \
                ((Ucl[k,i,sc]+Dcl[k,i,sc]+Fcl[k,i,sc])/Ncl[k,i,sc]) * (1 - (1-(1-e_condom)*beta_mf )**(n_cl_gf*p_cl_gf[i])          * (1-beta_mf)**(n_cl_gf*(1-p_cl_gf[i]))) + \
                (Tcl[k,i,sc]                         /Ncl[k,i,sc]) * (1 - (1-(1-e_art)*(1-e_condom)*beta_mf)**(n_cl_gf*p_cl_gf[i]) * (1-(1-e_art)*beta_mf)**(n_cl_gf * (1-p_cl_gf[i]))) + \
                ((Ugm[k,i,sc]+Dgm[k,i,sc]+Fgm[k,i,sc])/Ngm[k,i,sc]) * (1 - (1-(1-e_condom)*beta_mf )**(n_gm_gf*p_gm_gf[i])          * (1-beta_mf)**(n_gm_gf*(1-p_gm_gf[i]))) + \
                (Tgm[k,i,sc]                         /Ngm[k,i,sc]) * (1 - (1-(1-e_art)*(1-e_condom)*beta_mf)**(n_gm_gf*p_gm_gf[i]) * (1-(1-e_art)*beta_mf)**(n_gm_gf*(1-p_gm_gf[i])))

                force_of_infection_sw = \
                force_of_infection_gf + \
                ((Ucl[k,i,sc]+Dcl[k,i,sc]+Fcl[k,i,sc])/Ncl[k,i,sc]) * (1 - (1-(1-e_condom)*beta_mf)**(n_cl_sw*p_cl_sw[i])           * (1-beta_mf)**(n_cl_sw*(1-p_cl_sw[i]))) + \
                (Tcl[k,i,sc]                          /Ncl[k,i,sc]) * (1 - (1-(1-e_art)*(1-e_condom)*beta_mf)**(n_cl_sw*p_cl_sw[i]) * (1-(1-e_art)*beta_mf)**(n_cl_sw*(1-p_cl_sw[i])))

                force_of_infection_sw_i = \
                force_of_infection_gf + \
                ((Ucl[k,i,sc]+Dcl[k,i,sc]+Fcl[k,i,sc])/Ncl[k,i,sc]) * (1 - (1-(1-e_condom)*beta_mf*delta)**(n_cl_sw*p_cl_sw[i])           * (1-beta_mf*delta)**(n_cl_sw*(1-p_cl_sw[i]))) + \
                (Tcl[k,i,sc]                         /Ncl[k,i,sc]) * (1 -  (1-(1-e_art)*(1-e_condom)*beta_mf*delta)**(n_cl_sw*p_cl_sw[i]) * (1-(1-e_art)*beta_mf*delta)**(n_cl_sw*(1-p_cl_sw[i])))

                swclratio = (Nsw[k,i,sc] + Nsw_i[k,i,sc]) / Ncl[k,i,sc]
                force_of_infection_cl = \
                force_of_infection_gm + \
                ((Usw[k,i,sc]+Dsw[k,i,sc]+Fsw[k,i,sc]+Usw_i[k,i,sc]+Dsw_i[k,i,sc]+Fsw_i[k,i,sc])/(Nsw[k,i,sc]+Nsw_i[k,i,sc])) * (1 - (1-(1-e_condom)*beta_fm)**(n_cl_sw*swclratio*p_cl_sw[i])           * (1-beta_fm)**(n_cl_sw*swclratio*(1-p_cl_sw[i]))) + \
                ((Tsw[k,i,sc]                        +Tsw_i[k,i,sc])                            /(Nsw[k,i,sc]+Nsw_i[k,i,sc])) * (1 - (1-(1-e_art)*(1-e_condom)*beta_fm)**(n_cl_sw*swclratio*p_cl_sw[i]) * (1-(1-e_art)*beta_fm)**(n_cl_sw*swclratio*(1-p_cl_sw[i])))

                assert force_of_infection_sw<1, 'Invalid force of infection'

                total_births = b * (Nsw[k,i,sc] + Nsw_i[k,i,sc] + Ncl[k,i,sc] + Ngm[k,i,sc] + Ngf[k,i,sc])

                prop_w_sti = (Nsw_i[k,i,sc] / (Nsw[k,i,sc] + Nsw_i[k,i,sc]))
                prop_n_sti = (Nsw[k,i,sc]   / (Nsw[k,i,sc] + Nsw_i[k,i,sc]))

                totalpopulation = Nsw[k,i,sc] + Nsw_i[k,i,sc] + Ncl[k,i,sc] + Ngf[k,i,sc] + Ngm[k,i,sc]
                Ssw[k,j,sc]   = Ssw[k,i,sc]   - force_of_infection_sw*Ssw[k,i,sc]     - d*Ssw[k,i,sc]   + treat_sti*Ssw_i[k,i,sc] - alpha*Ssw[k,i,sc] + total_births*Nsw[k,i,sc]/totalpopulation
                Ssw_i[k,j,sc] = Ssw_i[k,i,sc] - force_of_infection_sw_i*Ssw_i[k,i,sc] - d*Ssw_i[k,i,sc] - treat_sti*Ssw_i[k,i,sc] + alpha*Ssw[k,i,sc] + total_births*Nsw_i[k,i,sc]/totalpopulation
                Scl[k,j,sc]   = Scl[k,i,sc]   - force_of_infection_cl*Scl[k,i,sc]     - d*Scl[k,i,sc] + total_births*Ncl[k,i,sc]/totalpopulation + clientrecylce[0]
                Sgm[k,j,sc]   = Sgm[k,i,sc]   - force_of_infection_gm*Sgm[k,i,sc]     - d*Sgm[k,i,sc] + total_births*Ngm[k,i,sc]/totalpopulation - clientrecylce[0]
                Sgf[k,j,sc]   = Sgf[k,i,sc]   - force_of_infection_gf*Sgf[k,i,sc]     - d*Sgf[k,i,sc] + total_births*Ngf[k,i,sc]/totalpopulation

                Usw[k,j,sc]   = Usw[k,i,sc]   + force_of_infection_sw*Ssw[k,i,sc]     - test_sw*Usw[k,i,sc]   - d_hiv*Usw[k,i,sc]   + treat_sti*Usw_i[k,i,sc] - alpha*Usw[k,i,sc]
                Usw_i[k,j,sc] = Usw_i[k,i,sc] + force_of_infection_sw_i*Ssw_i[k,i,sc] - test_sw*Usw_i[k,i,sc] - d_hiv*Usw_i[k,i,sc] - treat_sti*Usw_i[k,i,sc] + alpha*Usw[k,i,sc]
                Ucl[k,j,sc]   = Ucl[k,i,sc]   + force_of_infection_cl*Scl[k,i,sc]     - test_op*Ucl[k,i,sc]   - d_hiv*Ucl[k,i,sc] + clientrecylce[1]
                Ugm[k,j,sc]   = Ugm[k,i,sc]   + force_of_infection_gm*Sgm[k,i,sc]     - test_op*Ugm[k,i,sc]   - d_hiv*Ugm[k,i,sc] - clientrecylce[1]
                Ugf[k,j,sc]   = Ugf[k,i,sc]   + force_of_infection_gf*Sgf[k,i,sc]     - test_op*Ugf[k,i,sc]   - d_hiv*Ugf[k,i,sc]

                Dsw[k,j,sc]   = Dsw[k,i,sc]   + test_sw*Usw[k,i,sc]   - phi_sw*Dsw[k,i,sc]   + pi_sw*Tsw[k,i,sc]   - d_hiv*Dsw[k,i,sc]   + treat_sti*Dsw_i[k,i,sc] - alpha*Dsw[k,i,sc]
                Dsw_i[k,j,sc] = Dsw_i[k,i,sc] + test_sw*Usw_i[k,i,sc] - phi_sw*Dsw_i[k,i,sc] + pi_sw*Tsw_i[k,i,sc] - d_hiv*Dsw_i[k,i,sc] - treat_sti*Dsw_i[k,i,sc] + alpha*Dsw[k,i,sc]
                Dcl[k,j,sc]   = Dcl[k,i,sc]   + test_op*Ucl[k,i,sc]   - phi_cl*Dcl[k,i,sc]   + pi_cl*Tcl[k,i,sc]   - d_hiv*Dcl[k,i,sc]   + clientrecylce[2]
                Dgm[k,j,sc]   = Dgm[k,i,sc]   + test_op*Ugm[k,i,sc]   - phi_gm*Dgm[k,i,sc]   + pi_gm*Tgm[k,i,sc]   - d_hiv*Dgm[k,i,sc]   - clientrecylce[2]
                Dgf[k,j,sc]   = Dgf[k,i,sc]   + test_op*Ugf[k,i,sc]   - phi_gf*Dgf[k,i,sc]   + pi_gf*Tgf[k,i,sc]   - d_hiv*Dgf[k,i,sc]

                Tsw[k,j,sc]   = Tsw[k,i,sc]   + phi_sw*Dsw[k,i,sc]   - pi_sw*Tsw[k,i,sc]   - zeta*Tsw[k,i,sc]   - d_art*Tsw[k,i,sc]   + treat_sti*Tsw_i[k,i,sc] - alpha*Tsw[k,i,sc]
                Tsw_i[k,j,sc] = Tsw_i[k,i,sc] + phi_sw*Dsw_i[k,i,sc] - pi_sw*Tsw_i[k,i,sc] - zeta*Tsw_i[k,i,sc] - d_art*Tsw_i[k,i,sc] - treat_sti*Tsw_i[k,i,sc] + alpha*Tsw[k,i,sc]
                Tcl[k,j,sc]   = Tcl[k,i,sc]   + phi_cl*Dcl[k,i,sc]   - pi_cl*Tcl[k,i,sc]   - zeta*Tcl[k,i,sc]   - d_art*Tcl[k,i,sc]   + clientrecylce[3]
                Tgm[k,j,sc]   = Tgm[k,i,sc]   + phi_gm*Dgm[k,i,sc]   - pi_gm*Tgm[k,i,sc]   - zeta*Tgm[k,i,sc]   - d_art*Tgm[k,i,sc]   - clientrecylce[3]
                Tgf[k,j,sc]   = Tgf[k,i,sc]   + phi_gf*Dgf[k,i,sc]   - pi_gf*Tgf[k,i,sc]   - zeta*Tgf[k,i,sc]   - d_art*Tgf[k,i,sc]

                Fsw[k,j,sc]   = Fsw[k,i,sc]    + zeta*Tsw[k,i,sc]   - d_hiv*Fsw[k,i,sc]   + treat_sti*Fsw_i[k,i,sc] - alpha*Fsw[k,i,sc]
                Fsw_i[k,j,sc] = Fsw_i [k,i,sc] + zeta*Tsw_i[k,i,sc] - d_hiv*Fsw_i[k,i,sc] - treat_sti*Fsw_i[k,i,sc] + alpha*Fsw[k,i,sc]
                Fcl[k,j,sc]   = Fcl[k,i,sc]    + zeta*Tcl[k,i,sc]   - d_hiv*Fcl[k,i,sc] + clientrecylce[4]
                Fgm[k,j,sc]   = Fgm[k,i,sc]    + zeta*Tgm[k,i,sc]   - d_hiv*Fgm[k,i,sc] - clientrecylce[4]
                Fgf[k,j,sc]   = Fgf[k,i,sc]    + zeta*Tgf[k,i,sc]   - d_hiv*Fgf[k,i,sc]

                Nsw[k,j,sc]    = Ssw[k,j,sc]   + Usw[k,j,sc]   + Dsw[k,j,sc]   + Tsw[k,j,sc]   + Fsw[k,j,sc]
                Nsw_i[k,j,sc]  = Ssw_i[k,j,sc] + Usw_i[k,j,sc] + Dsw_i[k,j,sc] + Tsw_i[k,j,sc] + Fsw_i[k,j,sc]
                Ncl[k,j,sc]    = Scl[k,j,sc]   + Ucl[k,j,sc]   + Dcl[k,j,sc]   + Tcl[k,j,sc]   + Fcl[k,j,sc]
                Ngm[k,j,sc]    = Sgm[k,j,sc]   + Ugm[k,j,sc]   + Dgm[k,j,sc]   + Tgm[k,j,sc]   + Fgm[k,j,sc]
                Ngf[k,j,sc]    = Sgf[k,j,sc]   + Ugf[k,j,sc]   + Dgf[k,j,sc]   + Tgf[k,j,sc]   + Fgf[k,j,sc]

            prevalences_SW_Combined[k,j,sc]      = ((Usw[k,j,sc]   + Dsw[k,j,sc]   + Tsw[k,j,sc]   + Fsw[k,j,sc]   + Usw_i[k,j,sc] + Dsw_i[k,j,sc] + Tsw_i[k,j,sc] + Fsw_i[k,j,sc])   / (Nsw[k,j,sc] + Nsw_i[k,j,sc]))  * 100/100
            prevalences_SW[k,j,sc]    = ((Usw[k,j,sc] + Dsw[k,j,sc] + Tsw[k,j,sc] + Fsw[k,j,sc]) / Nsw[k,j,sc]) * 100/100
            prevalences_SW_I[k,j,sc]    = ((Usw_i[k,j,sc] + Dsw_i[k,j,sc] + Tsw_i[k,j,sc] + Fsw_i[k,j,sc]) / Nsw_i[k,j,sc]) * 100/100
            prevalences_CL[k,j,sc]      = ((Ucl[k,j,sc]   + Dcl[k,j,sc]   + Tcl[k,j,sc]   + Fcl[k,j,sc])   / Ncl[k,j,sc])   * 100/100
            prevalences_GM[k,j,sc]      = ((Ugm[k,j,sc]   + Dgm[k,j,sc]   + Tgm[k,j,sc]   + Fgm[k,j,sc])   / Ngm[k,j,sc])   * 100/100
            prevalences_GF[k,j,sc]      = ((Ugf[k,j,sc]   + Dgf[k,j,sc]   + Tgf[k,j,sc]   + Fgf[k,j,sc])   / Ngf[k,j,sc])   * 100/100

            assert prevalences_SW_Combined[k,j,sc]<1, 'Invalid prevalence'

print(f"Completed {uncertainty_simulations} uncertainty simulations.")

## Model results

In [ ]:
prevalence_arrays = {
    "Female sex workers": prevalences_SW_Combined[:, :, 0] * 100,
    "Clients": prevalences_CL[:, :, 0] * 100,
    "General males": prevalences_GM[:, :, 0] * 100,
    "General females": prevalences_GF[:, :, 0] * 100,
}

summary_frames = []
for population, simulations in prevalence_arrays.items():
    summary_frames.append(pd.DataFrame({
        "Year": years,
        "Population": population,
        "Minimum prevalence (%)": simulations.min(axis=0),
        "Maximum prevalence (%)": simulations.max(axis=0),
        "Average prevalence (%)": simulations.mean(axis=0),
    }))

prevalence_results = pd.concat(summary_frames, ignore_index=True)
endpoint_summary = (
    prevalence_results[prevalence_results["Year"].isin([years[0], years[-1]])]
    .pivot(index="Population", columns="Year", values="Average prevalence (%)")
    .rename(columns={years[0]: f"{years[0]} average (%)", years[-1]: f"{years[-1]} average (%)"})
)
endpoint_summary["Absolute change (percentage points)"] = (
    endpoint_summary[f"{years[-1]} average (%)"] - endpoint_summary[f"{years[0]} average (%)"]
)
endpoint_summary["Relative change (%)"] = (
    endpoint_summary["Absolute change (percentage points)"]
    / endpoint_summary[f"{years[0]} average (%)"] * 100
)

display(endpoint_summary.sort_values(f"{years[0]} average (%)", ascending=False))

### Results

The average modelled prevalence declines in every population between 2000 and 2030. Female sex workers remain the most affected group, although their average prevalence falls from 37.54 to 2.84 percent. Client prevalence falls from 10.74 to 0.63 percent. Average prevalence among general males falls from 5.69 to 0.30 percent, while the estimate for general females falls from 4.73 to 0.38 percent.

The relative decline ranges from 91.96 percent among general females to 94.72 percent among general males. These reductions describe what happens when the original 2013 assumptions remain in place through 2030. They do not include later policy changes, updated treatment guidelines, new prevention technologies, or disruptions in service delivery.

## Prevalence trends and simulation ranges

The blue line is the average across uncertainty simulations. The shaded area is the full minimum to maximum range. The red points are the observed prevalence values included in the original model.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8.5), sharex=True)
axes = axes.flatten()

for ax, (population, simulations) in zip(axes, prevalence_arrays.items()):
    average = simulations.mean(axis=0)
    minimum = simulations.min(axis=0)
    maximum = simulations.max(axis=0)
    observed = observed_prevalence[population]

    ax.fill_between(years, minimum, maximum, color="#4C78A8", alpha=0.18,
                    label="Simulation range")
    ax.plot(years, average, color="#1F77B4", linewidth=2.5,
            label="Model average")
    ax.scatter(observed[:, 0], observed[:, 1], s=65, color="#E45756",
               edgecolor="white", linewidth=1.0, zorder=3, label="Observed data")

    ax.set_title(population, fontsize=13, pad=10)
    ax.set_xlim(years[0], years[-1])
    ax.set_ylim(bottom=0)
    ax.set_xticks(np.arange(years[0], years[-1] + 1, 5))
    ax.xaxis.set_major_formatter(FormatStrFormatter("%d"))
    ax.grid(True, alpha=0.55)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle("HIV prevalence: model estimates and observed data",
             fontsize=17, fontweight="bold", y=0.97)
fig.supylabel("HIV prevalence (%)", fontsize=12, fontweight="semibold", x=0.025)
fig.supxlabel("Year", fontsize=12, fontweight="semibold", y=0.105)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 0.025),
           ncol=3, frameon=False, columnspacing=2.2, handlelength=2.2)
fig.subplots_adjust(left=0.08, right=0.98, top=0.90, bottom=0.19,
                    hspace=0.28, wspace=0.12)

plt.show()

**Interpretation.** Female sex workers have the highest modelled prevalence throughout the period, followed by clients. The average model curves decline in every group, but every supplied observed point is above the corresponding average curve. The shaded ranges show sensitivity to the sampled parameter values and should not be interpreted as confidence intervals.

## Descriptive calibration diagnostics

The following comparison evaluates the average model curve at each observed year. Residuals are calculated as observed prevalence minus modelled prevalence. Positive residuals therefore indicate that the model is below the observed point.

These statistics describe fit to the small set of supplied observations. They are not a formal validation study.

In [ ]:
calibration_rows = []

for population, observed in observed_prevalence.items():
    model_average = prevalence_arrays[population].mean(axis=0)
    predicted = np.interp(observed[:, 0], years, model_average)

    for (year, observed_value), predicted_value in zip(observed, predicted):
        calibration_rows.append({
            "Population": population,
            "Year": int(year),
            "Observed prevalence (%)": observed_value,
            "Model average (%)": predicted_value,
            "Residual (percentage points)": observed_value - predicted_value,
        })

calibration_comparison = pd.DataFrame(calibration_rows)

fit_metrics = pd.DataFrame([
    {
        "Population": population,
        "Observed points": len(group),
        "Mean absolute error (percentage points)":
            np.mean(np.abs(group["Residual (percentage points)"])),
        "Root mean squared error (percentage points)":
            np.sqrt(np.mean(group["Residual (percentage points)"] ** 2)),
        "Mean residual (percentage points)":
            np.mean(group["Residual (percentage points)"]),
    }
    for population, group in calibration_comparison.groupby("Population", sort=False)
])

display(calibration_comparison)
display(fit_metrics)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

metric_plot = fit_metrics.sort_values(
    "Mean absolute error (percentage points)", ascending=True
)

ax.barh(
    metric_plot["Population"],
    metric_plot["Mean absolute error (percentage points)"],
    color="#4C78A8",
    edgecolor="#2F4B63",
)
ax.set_title("Calibration error by population group", fontsize=14, pad=12)
ax.set_xlabel("Mean absolute error (percentage points)")
ax.grid(axis="x", alpha=0.55)
ax.grid(axis="y", visible=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for patch, value in zip(ax.patches, metric_plot["Mean absolute error (percentage points)"]):
    ax.text(value + 0.08, patch.get_y() + patch.get_height() / 2,
            f"{value:.2f}", va="center", fontsize=10)

plt.tight_layout()
plt.show()

**Interpretation.** The largest calibration gap is among female sex workers, with a mean absolute error of 9.86 percentage points. Error is smaller but still consistently positive in the other groups. This pattern indicates systematic underestimation of the supplied observations rather than errors that balance around zero.

## Modelled HIV programme states in 2030

This view summarizes the average distribution of modelled people living with HIV across the undiagnosed, diagnosed, treated, and failed treatment states in the final simulation year. It is a model output, not an observed treatment cascade.

In [ ]:
final_index = -1

cascade_counts = {
    "Female sex workers": {
        "Undiagnosed": np.mean(Usw[:, final_index, 0] + Usw_i[:, final_index, 0]),
        "Diagnosed": np.mean(Dsw[:, final_index, 0] + Dsw_i[:, final_index, 0]),
        "On treatment": np.mean(Tsw[:, final_index, 0] + Tsw_i[:, final_index, 0]),
        "Failed treatment": np.mean(Fsw[:, final_index, 0] + Fsw_i[:, final_index, 0]),
    },
    "Clients": {
        "Undiagnosed": np.mean(Ucl[:, final_index, 0]),
        "Diagnosed": np.mean(Dcl[:, final_index, 0]),
        "On treatment": np.mean(Tcl[:, final_index, 0]),
        "Failed treatment": np.mean(Fcl[:, final_index, 0]),
    },
    "General males": {
        "Undiagnosed": np.mean(Ugm[:, final_index, 0]),
        "Diagnosed": np.mean(Dgm[:, final_index, 0]),
        "On treatment": np.mean(Tgm[:, final_index, 0]),
        "Failed treatment": np.mean(Fgm[:, final_index, 0]),
    },
    "General females": {
        "Undiagnosed": np.mean(Ugf[:, final_index, 0]),
        "Diagnosed": np.mean(Dgf[:, final_index, 0]),
        "On treatment": np.mean(Tgf[:, final_index, 0]),
        "Failed treatment": np.mean(Fgf[:, final_index, 0]),
    },
}

cascade_counts_df = pd.DataFrame.from_dict(cascade_counts, orient="index")
cascade_share_df = cascade_counts_df.div(cascade_counts_df.sum(axis=1), axis=0) * 100

display(cascade_share_df.rename_axis("Population").round(2))

colors = ["#4C78A8", "#F2CF5B", "#59A14F", "#E15759"]
ax = cascade_share_df.plot(
    kind="barh", stacked=True, figsize=(10, 5.5), color=colors, width=0.68
)
ax.set_title("Modelled HIV programme state distribution in 2030", fontsize=14, pad=12)
ax.set_xlabel("Share of modelled people living with HIV (%)")
ax.set_ylabel("")
ax.set_xlim(0, 100)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=4, frameon=False)
ax.grid(axis="x", alpha=0.55)
ax.grid(axis="y", visible=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

**Interpretation.** In the modelled 2030 distribution, treatment represents 60.09 percent of people living with HIV among clients, 57.37 percent among general males, 56.75 percent among general females, and 42.21 percent among female sex workers. Female sex workers also retain the largest diagnosed but untreated share at 35.09 percent. These values are simulated programme states and should not be read as observed national treatment coverage.

## Conclusion

This historical model treats HIV transmission as a connected system. Infection risk in one population affects other populations through the specified mixing pathways, while testing, treatment, adherence, condom use, and sexually transmitted infection status alter the projected course.

The retained assumptions produce declines of 91.96 to 94.72 percent in average prevalence across the four reported populations between 2000 and 2030. The observed points are consistently higher than the average model curves. The largest calibration gap is among female sex workers, where mean absolute error is 9.86 percentage points. The public indicators also show why the projection should remain separate from current national estimates: they cover different populations and use newer surveillance and estimation methods.

The notebook provides a reproducible record of the analytical methods developed through the World Bank-supported University of New South Wales training and the Technical Working Group engagement. A current policy model would require updated behavioural and programme data, formal calibration to recent surveillance, sensitivity analysis of the extended period, and independent validation.

## References

1. National Agency for the Control of AIDS. (2013). *Report on the final mathematical modelling training course* [Unpublished workshop report]. Abuja, Nigeria.

2. Centers for Disease Control and Prevention. (2025). *HIV and TB overview: Nigeria*. U.S. Department of Health and Human Services. https://www.cdc.gov/global-hiv-tb/php/where-we-work/nigeria.html

3. Joint United Nations Programme on HIV/AIDS. (2019, March 14). *New survey results indicate that Nigeria has an HIV prevalence of 1.4%* [Press release]. https://www.unaids.org/en/resources/presscentre/pressreleaseandstatementarchive/2019/march/20190314_nigeria

4. Joint United Nations Programme on HIV/AIDS. (2026). *Nigeria country profile*. https://www.unaids.org/en/regionscountries/countries/nigeria

5. World Bank. (2025). *Prevalence of HIV, total (% of population ages 15–49) – Nigeria* [Data set]. World Development Indicators. https://data.worldbank.org/indicator/SH.DYN.AIDS.ZS?locations=NG

6. World Bank. (2025). *Incidence of HIV, ages 15–49 (per 1,000 uninfected population ages 15–49) – Nigeria* [Data set]. World Development Indicators. https://data.worldbank.org/indicator/SH.HIV.INCD.ZS?locations=NG

## Reproducibility

The notebook requires Python 3 with NumPy, pandas, Matplotlib, IPython, and Jupyter. The cells are arranged in execution order, and every table and figure appears directly below the code that produces it. The random seed is fixed at zero so the uncertainty simulation can be reproduced.

Changes to parameter ranges, equations, population definitions, calibration data, or intervention assumptions should be documented before the notebook is reused for research or programme planning.